# Appendix Tables: Top Triple Occurrences per TKGU Operation

For each TKGU operation type, shows the most frequent triples in the dataset
with example passages. Emerging entities are shown in **bold**.

Produces one LaTeX `tabularx` table per TKGU operation (Exists, Add, Mint+Add, Infer, Deprecate).

**Data source:** Complete dataset from old pipeline pkl (until full-dataset pipeline is ported).

In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(os.path.abspath('../../..'), 'src'))

In [2]:
# --- CONFIGURE THIS ---

# Complete dataset (old pipeline pkls)
# TODO: replace with refactored pkl once full-dataset pipeline is ported
COMPLETE_DATASET_DIR = '/path/to/storage/emerge/output/experiments/dataset_stats_pkls/20260124_all_dataset_no_models_v13_all/'

# Number of top triples to show per TKGU operation
TOP_N = 5

# Assessor filter for complete dataset
ASSESSOR_MAP = {
    'triple_assertion': 'Meta-Llama-3.1-8B_triple_assertion',
    'triple_deprecation': 'Meta-Llama-3.1-405B_triple_deprecation'
}

In [3]:
# Load complete dataset
df_gt = pd.read_pickle(os.path.join(COMPLETE_DATASET_DIR, 'df_wiki_predictions_and_gt_cie.pkl'))
df_inst = pd.read_pickle(os.path.join(COMPLETE_DATASET_DIR, 'df_instances_v13.pkl'))

# Join passage and snapshot_year
df = df_gt.merge(
    df_inst[['hash_id', 'passage', 'snapshot_year', 'delta_weeks']],
    on='hash_id', how='inner'
)

print(f'Loaded {len(df)} rows, {df.hash_id.nunique()} unique instances')
print(f'tkgu_types: {df.tkgu_type.unique().tolist()}')

Loaded 3887196 rows, 233113 unique instances
tkgu_types: ['ee-triples', 'x-triples', 'ee-kg-triples', 'e-triples', 'd-triples']


In [4]:
# Filter by assessor (keep only LLM-supported triples)
expected = df['prompt_type'].map(ASSESSOR_MAP)
mask_expected = df['llm_assessor'].astype(str) == expected.astype(str)
mask_bool = df['llm_assessor_result'].apply(lambda x: isinstance(x, (bool, np.bool_)))
mask_true = df['llm_assessor_result'] == True

df_filtered = df[mask_expected & mask_bool & mask_true].copy()
print(f'After assessor filter: {len(df_filtered)} rows')

After assessor filter: 1452296 rows


In [5]:
# Format triple with emerging entities in bold
def format_triple(row):
    head = row['triple_head_label']
    tail = row['triple_tail_label']
    if row.get('triple_emerging_head', False):
        head = f'\\textbf{{{head}}}'
    if row.get('triple_emerging_tail', False):
        tail = f'\\textbf{{{tail}}}'
    return f'$\\langle${head}; {row["triple_relation_label"]}; {tail}$\\rangle$'

df_filtered['triple_repr'] = df_filtered.apply(format_triple, axis=1)

# TKGU type config
tkgu_types = {
    'x-triples': {'latex_cmd': r'\opexists', 'label': 'Exists'},
    'e-triples': {'latex_cmd': r'\opadd', 'label': 'Add'},
    'ee-triples': {'latex_cmd': r'\opmintadd', 'label': 'Mint+Add'},
    'ee-kg-triples': {'latex_cmd': r'\opinfer', 'label': 'Infer'},
    'd-triples': {'latex_cmd': r'\opdeprecate', 'label': 'Deprecate'},
}

In [6]:
# Compute top triples per TKGU type and generate LaTeX

for tkgu_type, tkgu_info in tkgu_types.items():
    df_type = df_filtered[df_filtered['tkgu_type'] == tkgu_type].copy()
    if df_type.empty:
        print(f'\n% No data for {tkgu_type}\n')
        continue

    # Count occurrences of each triple (across all passages/instances)
    grouped = (
        df_type.groupby(['snapshot_year', 'triple_repr', 'triple_head', 'triple_relation', 'triple_tail'])
        .size()
        .reset_index(name='count')
    )

    # Top N overall (sorted by count descending)
    top = grouped.nlargest(TOP_N, 'count')

    # Find shortest passage for each top triple
    latex_rows = []
    for _, row in top.iterrows():
        passages = df_type[
            (df_type['triple_head'] == row['triple_head']) &
            (df_type['triple_relation'] == row['triple_relation']) &
            (df_type['triple_tail'] == row['triple_tail'])
        ]['passage'].drop_duplicates()

        shortest = passages.loc[passages.str.len().idxmin()] if len(passages) > 0 else ''

        latex_rows.append(
            f"{int(row['snapshot_year'])} & {int(row['count'])} & {row['triple_repr']} & {shortest} \\\\"
        )

    body = '\n'.join(latex_rows)
    ee_note = ' (emerging entities in bold)' if tkgu_type in ('ee-triples', 'ee-kg-triples') else ''

    latex = f"""\\renewcommand{{\\arraystretch}}{{1.5}}
\\begin{{table}}[H]
\\centering
\\small
\\caption{{Example entries of the most frequent {tkgu_info['latex_cmd']}~TKGU operation instances in \\datasetname{ee_note}, showing the snapshot (Snap.), triple, and number of instances (\\#).}}
\\label{{tab:{tkgu_type}-examples}}
\\begin{{tabularx}}{{\\linewidth}}{{l r >{{\\hsize=0.8\\hsize}}X >{{\\hsize=1.2\\hsize}}X}}
\\toprule
\\textbf{{Snap.}} & \\textbf{{\\#}} & \\textbf{{Triple}} & \\textbf{{Example Passage}} \\\\
\\midrule
{body}
\\bottomrule
\\end{{tabularx}}
\\end{{table}}
\\renewcommand{{\\arraystretch}}{{1}}"""

    print(latex)
    print()

\renewcommand{\arraystretch}{1.5}
\begin{table}[H]
\centering
\small
\caption{Example entries of the most frequent \opexists~TKGU operation instances in \datasetname, showing the snapshot (Snap.), triple, and number of instances (\#).}
\label{tab:x-triples-examples}
\begin{tabularx}{\linewidth}{l r >{\hsize=0.8\hsize}X >{\hsize=1.2\hsize}X}
\toprule
\textbf{Snap.} & \textbf{\#} & \textbf{Triple} & \textbf{Example Passage} \\
\midrule
2021 & 834 & $\langle$Donald Trump; candidacy in election; 2020 United States presidential election$\rangle$ & Excerpt from the Wikipedia page describing Bernadine Kent: In September 2020, Kennedy Kent endorsed Republican President Donald Trump for reelection over Democratic nominee Joe Biden. \\
2021 & 827 & $\langle$2020 United States presidential election; candidate; Donald Trump$\rangle$ & Excerpt from the Wikipedia page describing Bernadine Kent: In September 2020, Kennedy Kent endorsed Republican President Donald Trump for reelection over Democratic 

\renewcommand{\arraystretch}{1.5}
\begin{table}[H]
\centering
\small
\caption{Example entries of the most frequent \opadd~TKGU operation instances in \datasetname, showing the snapshot (Snap.), triple, and number of instances (\#).}
\label{tab:e-triples-examples}
\begin{tabularx}{\linewidth}{l r >{\hsize=0.8\hsize}X >{\hsize=1.2\hsize}X}
\toprule
\textbf{Snap.} & \textbf{\#} & \textbf{Triple} & \textbf{Example Passage} \\
\midrule
2021 & 811 & $\langle$Donald Trump; participant in; 2020 United States presidential election$\rangle$ & Excerpt from the Wikipedia page describing Bernadine Kent: In September 2020, Kennedy Kent endorsed Republican President Donald Trump for reelection over Democratic nominee Joe Biden. \\
2021 & 635 & $\langle$2020 United States presidential election; successful candidate; Joe Biden$\rangle$ & Excerpt from the Wikipedia page describing Curse of Tippecanoe: The winner of the 2020 election, President Joe Biden, is next in line to this purported curse. \\
2021 

\renewcommand{\arraystretch}{1.5}
\begin{table}[H]
\centering
\small
\caption{Example entries of the most frequent \opmintadd~TKGU operation instances in \datasetname (emerging entities in bold), showing the snapshot (Snap.), triple, and number of instances (\#).}
\label{tab:ee-triples-examples}
\begin{tabularx}{\linewidth}{l r >{\hsize=0.8\hsize}X >{\hsize=1.2\hsize}X}
\toprule
\textbf{Snap.} & \textbf{\#} & \textbf{Triple} & \textbf{Example Passage} \\
\midrule
2021 & 848 & $\langle$\textbf{January 6 United States Capitol attack}; significant person; Donald Trump$\rangle$ & Excerpt from the Wikipedia page describing Ilhan Omar: She called for the impeachment of President Donald Trump, in wake of the 2021 storming of the United States Capitol. \\
2020 & 670 & $\langle$Qasem Soleimani; significant event; \textbf{assassination of Qasem Soleimani}$\rangle$ & Excerpt from the Wikipedia page describing Abu Mahdi al-Muhandis: He was killed by a targeted U.S. drone strike at the Baghdad Inte

\renewcommand{\arraystretch}{1.5}
\begin{table}[H]
\centering
\small
\caption{Example entries of the most frequent \opinfer~TKGU operation instances in \datasetname (emerging entities in bold), showing the snapshot (Snap.), triple, and number of instances (\#).}
\label{tab:ee-kg-triples-examples}
\begin{tabularx}{\linewidth}{l r >{\hsize=0.8\hsize}X >{\hsize=1.2\hsize}X}
\toprule
\textbf{Snap.} & \textbf{\#} & \textbf{Triple} & \textbf{Example Passage} \\
\midrule
2021 & 3455 & $\langle$\textbf{January 6 United States Capitol attack}; country; United States of America$\rangle$ & Excerpt from the Wikipedia page describing List of riots: ====2021==== *2021 – January 6-7: Trump supporters infiltrated Capitol Hill in Washington DC., 5 people killed. \\
2021 & 3308 & $\langle$\textbf{January 6 United States Capitol attack}; part of; 2020–2021 United States election protests$\rangle$ & Excerpt from the Wikipedia page describing List of riots: ====2021==== *2021 – January 6-7: Trump supporter